# Advanced Features for Production Systems

`OnlineChangeDetector` wraps the low-level BOCPD core with production niceties:
probability thresholds, cooldown logic, metadata tracking, and utilities to
inspect detected segments. This notebook showcases those pieces.


## 0. Environment setup


In [1]:
import sys
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD if (CWD / "fast_bocpd").exists() else CWD.parent
EXAMPLES_DIR = REPO_ROOT / "examples"

sys.path.append(str(REPO_ROOT))
sys.path.append(str(EXAMPLES_DIR))


## 1. Imports and helpers


In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from fast_bocpd import BOCPD, ConstantHazard, GaussianNIG
from fast_bocpd.utils import OnlineChangeDetector

from _helpers import generate_piecewise_gaussian, plot_series_with_cp, seed_everything


### Utility helpers


In [3]:
seed_everything(11)

def stream_with_metadata(detector, values, timestamps):
    """Run OnlineChangeDetector and capture every emitted Changepoint."""
    records = []
    for ts, value in zip(timestamps, values):
        cp = detector.update(float(value), metadata={"timestamp": ts})
        if cp:
            records.append(cp)
    return records


def run_bocpd_trace(values, obs_model, hazard, max_run_length=600):
    """Run bare BOCPD to collect cp_prob and MAP run length for plotting."""
    values = np.asarray(values)
    bocpd = BOCPD(obs_model=obs_model, hazard=hazard, max_run_length=max_run_length)
    cp_probs = []
    map_r = []
    for value in values:
        _, cp_prob = bocpd.update(float(value))
        cp_probs.append(cp_prob)
        map_r.append(bocpd.get_map_run_length())
    return pd.DataFrame({
        "t": np.arange(len(values)),
        "value": values,
        "cp_prob": cp_probs,
        "map_r": map_r,
    })


## 2. Auto-configuration from the hazard

`OnlineChangeDetector` inspects the hazard to derive principled defaults for
`min_cp_prob`, `drop_prev_min`, and `cooldown`. The defaults depend on the
expected segment length (`lambda`) and a Bayes-factor style evidence threshold.


In [4]:
obs_model = GaussianNIG(mu0=0.0, kappa0=0.8, alpha0=0.3, beta0=0.3)
hazard = ConstantHazard(lambda_=180)
bocpd = BOCPD(obs_model=obs_model, hazard=hazard, max_run_length=600)

detector = OnlineChangeDetector(
    bocpd,
    min_cp_prob=None,
    drop_prev_min=None,
    cooldown=None,
    bayes_factor=5.0,
)

defaults = pd.DataFrame(
    {
        "parameter": ["min_cp_prob", "drop_prev_min", "cooldown", "reset_r"],
        "value": [
            detector.min_cp_prob,
            detector.drop_prev_min,
            detector.cooldown,
            detector.reset_r,
        ],
        "meaning": [
            "Probability threshold computed from hazard + Bayes factor",
            "Needed previous MAP run length before reset heuristic triggers",
            "Samples to wait before emitting another changepoint",
            "MAP run length considered a fresh start",
        ],
    }
)
defaults


,parameter,value,meaning
0,min_cp_prob,0.027174,Probability threshold computed from hazard + B...
1,drop_prev_min,45.000000,Needed previous MAP run length before reset he...
2,cooldown,18.000000,Samples to wait before emitting another change...
3,reset_r,2.000000,MAP run length considered a fresh start


The Bayes factor lets you dial the trade-off between sensitivity and false positives. Larger values demand stronger evidence before the probability trigger fires.


## 3. Streaming with metadata and automatic changepoints

Simulate three regimes, attach timestamps, and inspect the emitted
`Changepoint` dataclasses. Metadata is preserved, so downstream systems can
trace alerts back to precise records.


In [5]:
df, cps = generate_piecewise_gaussian(
    lengths=[160, 190, 150],
    means=[0.0, 1.0, -0.3],
    sigma=0.25,
    seed=22,
)
timestamps = pd.date_range("2024-01-01", periods=len(df), freq="min")

bocpd_stream = BOCPD(obs_model=obs_model, hazard=hazard, max_run_length=600)
detector_stream = OnlineChangeDetector(bocpd_stream, bayes_factor=5.0)

changepoints = stream_with_metadata(detector_stream, df["value"], timestamps)
cp_df = pd.DataFrame(
    [
        {
            "index": cp.index,
            "timestamp": cp.metadata["timestamp"],
            "prev_run": cp.prev_run_length,
            "cp_prob": cp.cp_prob,
            "map_run": cp.map_run_length,
        }
        for cp in changepoints
    ]
)
cp_df


,index,timestamp,prev_run,cp_prob,map_run
0,153,2024-01-01 02:33:00,153,0.035186,154
1,160,2024-01-01 02:40:00,160,0.592106,0
2,350,2024-01-01 05:50:00,189,0.880475,0
3,427,2024-01-01 07:07:00,76,0.050864,77
4,480,2024-01-01 08:00:00,129,0.112129,130


The `Changepoint` dataclass is printable and keeps the raw observation plus MAP confidence:


In [6]:
for cp in changepoints:
    print(cp)


Changepoint at t=153 ({'timestamp': Timestamp('2024-01-01 02:33:00')}): previous segment lasted 153 steps (P(CP)=3.5%, MAP r=154)
Changepoint at t=160 ({'timestamp': Timestamp('2024-01-01 02:40:00')}): previous segment lasted 160 steps (P(CP)=59.2%, MAP r=0)
Changepoint at t=350 ({'timestamp': Timestamp('2024-01-01 05:50:00')}): previous segment lasted 189 steps (P(CP)=88.0%, MAP r=0)
Changepoint at t=427 ({'timestamp': Timestamp('2024-01-01 07:07:00')}): previous segment lasted 76 steps (P(CP)=5.1%, MAP r=77)
Changepoint at t=480 ({'timestamp': Timestamp('2024-01-01 08:00:00')}): previous segment lasted 129 steps (P(CP)=11.2%, MAP r=130)


### Visual inspection


In [7]:
trace_df = run_bocpd_trace(df["value"], obs_model, hazard, max_run_length=600)
fig = plot_series_with_cp(
    trace_df,
    value_col="value",
    cp_prob_col="cp_prob",
    changepoints=cps,
    title="Detector output with emitted changepoints",
)
if not cp_df.empty:
    fig.add_trace(
        go.Scatter(
            x=cp_df["index"],
            y=trace_df.loc[cp_df["index"], "value"],
            mode="markers",
            marker=dict(color="#E15759", size=10, symbol="x"),
            name="Detected changepoint",
        )
    )
fig


`map_run` in the table above corresponds to `detector_stream.get_map_history()[index]`, so you can correlate alerts with posterior run-length estimates.


## 4. Inspecting segments and MAP history

`OnlineChangeDetector` accumulates MAP run-length history and segment indices so
you can extract aggregates without replaying the stream.


In [8]:
segments = detector_stream.get_segments()
segments_df = pd.DataFrame(
    {
        "segment": np.arange(len(segments)),
        "start": [s for s, _ in segments],
        "end": [e for _, e in segments],
        "length": [e - s for s, e in segments],
    }
)
segments_df


,segment,start,end,length
0,0,0,153,153
1,1,153,160,7
2,2,160,350,190
3,3,350,427,77
4,4,427,480,53
5,5,480,500,20


In [9]:
map_history = detector_stream.get_map_history()
pd.DataFrame({"t": np.arange(len(map_history)), "map_run": map_history}).head()


,t,map_run
0,0,1
1,1,2
2,2,3
3,3,4
4,4,5


## 5. Cooldown, drop_prev_min, and reset heuristics

Two signals trigger detections:
1. `cp_prob` crossing `min_cp_prob`.
2. MAP run length dropping from at least `drop_prev_min` down to `reset_r`.

`cooldown` debounces repeated alerts. The table below shows how different
settings affect the number of emitted changepoints on noisy data.


In [10]:
def count_changepoints(*, cooldown, drop_prev_min):
    bocpd_tmp = BOCPD(obs_model=obs_model, hazard=hazard, max_run_length=600)
    detector_tmp = OnlineChangeDetector(
        bocpd_tmp,
        cooldown=cooldown,
        drop_prev_min=drop_prev_min,
        min_cp_prob=detector_stream.min_cp_prob,
        reset_r=detector_stream.reset_r,
    )
    cps_tmp = stream_with_metadata(detector_tmp, df["value"], timestamps)
    return len(cps_tmp)

sweep = pd.DataFrame([
    {
        "cooldown": cd,
        "drop_prev_min": drop,
        "detections": count_changepoints(cooldown=cd, drop_prev_min=drop),
    }
    for cd in [0, detector_stream.cooldown, 30]
    for drop in [detector_stream.drop_prev_min, 5, 40]
])
sweep


,cooldown,drop_prev_min,detections
0,0,45,5
1,0,5,5
2,0,40,5
3,18,45,5
4,18,5,5
5,18,40,5
6,30,45,5
7,30,5,5
8,30,40,5


Short cooldowns and tiny `drop_prev_min` values make the detector fire rapidly. Increase them when you need to suppress duplicate alerts across noisy transitions.


## 6. State management and reset

Use `detector.reset()` to wipe history, or snapshot state with your own
serializer. After a reset the cooldown timer is cleared so the next detection
can fire immediately.


In [11]:
print(f"Changepoints before reset: {len(detector_stream.get_changepoints())}")
detector_stream.reset()
print(f"After reset: {len(detector_stream.get_changepoints())}")
print(f"MAP history cleared? {len(detector_stream.get_map_history()) == 0}")


Changepoints before reset: 5
After reset: 0
MAP history cleared? True


## 7. Packaging into a monitoring helper

A thin wrapper adds logging, batching, or alert delivery. The example below
exposes a `process` method and surfaces detections via a callback.


In [12]:
class ChangePointMonitor:
    def __init__(self, lambda_=150, bayes_factor=5.0, max_run_length=600):
        model = GaussianNIG(mu0=0.0, kappa0=1.0, alpha0=0.2, beta0=0.2)
        hazard = ConstantHazard(lambda_)
        bocpd = BOCPD(model, hazard, max_run_length=max_run_length)
        self.detector = OnlineChangeDetector(bocpd, bayes_factor=bayes_factor)

    def process(self, value, metadata=None):
        cp = self.detector.update(float(value), metadata=metadata)
        if cp:
            self.handle_changepoint(cp)

    def handle_changepoint(self, cp):
        print(f"ALERT: {cp}")

monitor = ChangePointMonitor()
for value in df["value"].iloc[:50]:
    monitor.process(value)


## Summary

- `OnlineChangeDetector` auto-tunes thresholds from the hazard and exposes both
probabilistic and run-length-based triggers.
- Metadata travels with each `Changepoint`, making downstream auditing easy.
- Utilities such as `get_segments`, `get_map_history`, `reset`, and
`cooldown`/`drop_prev_min` controls help integrate Fast-BOCPD into production
pipelines.
